<h1>Get conserved clusters inter species<h1/>

<h2> Import librairies <h2/>

In [1]:
import pymongo
from pymongo import mongo_client
import pandas as pd

<h2>Connect to MongoDB

In [55]:
from pymongo import mongo_client


MongoClient = pymongo.MongoClient("localhost", 27017)


GAM_w15_ncc_db = "GAM_w15_ncc"   
GANM_w25_ncc_db = "GANM_w25_ncc" 
LPM_w15_ncc_db = "LPM_w15_ncc"   
LPNM_w20_ncc_db = "LPNM_w20_ncc"   
OPM_w30_ncc_db = "OPM_w30_ncc"   
OPNM_w20_ncc_db = "OPNM_w20_ncc" 

conserved_cluster_inter = "Conserved_cluster_inter"
family='inter_specie_ncc'

ConcervedINTRA = MongoClient[conserved_cluster_inter]
GAMDB = MongoClient[GAM_w15_ncc_db]
GANMDB = MongoClient[GANM_w25_ncc_db]
LPMDB = MongoClient[LPM_w15_ncc_db]
LPNMDB = MongoClient[LPNM_w20_ncc_db]
OPMDB = MongoClient[OPM_w30_ncc_db]
OPNMDB = MongoClient[OPNM_w20_ncc_db]


<h2> extract conserved cluster inter species without ids

In [ ]:
for db1 in [GAMDB,GANMDB,LPMDB,LPNMDB,OPMDB,OPNMDB]:
    species1 = db1.list_collection_names()
    for specie in species1:
        collections = db1.get_collection(specie)
        documents = collections.find()
        for item in documents:
            cluster = item["cluster"]
            for db in [GAMDB,GANMDB,LPMDB,LPNMDB,OPMDB,OPNMDB]:
                for specie2 in db.list_collection_names():
                    if specie.lower() != specie2.lower():
                        col=db.get_collection(specie2)
                        cnt = col.count_documents(
                            {"cluster": cluster, "specie": {"$ne": specie2}})
                    if cnt>0:
                        print(cluster)
                        print(cnt,specie,str(specie2))
                        collec = ConcervedINTRA.get_collection(family)
                        if collec.find_one({"cluster": cluster}):
                            print("cluster exist")
                            collec.update_one(
                            {"cluster": cluster},
                            { "$addToSet": { "species": specie } }
                            )

                        else:
                            print("new doc")
                            entry = {"cluster":cluster, "species": [specie]}
                            collec.insert_one(entry)